# Budget, Genre, and Beyond: What Actually Drives a Film's Financial and Critical Success?

## Permissions
* [  ] YES - make available
* [X] NO - keep private

## Link to video
https://link.to.your.publicly.viewable.video

## Abstract

This project investigates how production budget, genre, country of production, and release era influence a film's financial performance (box office gross) and critical reception (IMDb score). Using two complementary publicly available datasets covering thousands of films released between 1986 and 2017, we merged, cleaned, and analyzed records to examine these relationships across multiple dimensions. After performing exploratory data analysis, we conducted two inferential analyses: an OLS multiple regression model predicting log gross revenue, and a Welch's t-test comparing the gross revenues of US versus non-US productions.

Our results confirm that production budget is the single strongest predictor of box office revenue, with a log-log correlation of approximately 0.67 and a statistically significant regression coefficient. Genre also plays a meaningful role — action, animation, and adventure films earn the highest revenues, while drama and biography earn far less despite receiving higher IMDb ratings. Critically, we find essentially no relationship between production budget and IMDb score (r ≈ 0.00), suggesting that financial investment does not translate into audience satisfaction. US films earn significantly higher gross revenues than non-US films (p < 0.001), consistent with Hollywood's established global distribution advantage. The budget-revenue relationship did not weaken post-2015 as hypothesized; instead, a slight strengthening likely reflects a selection effect where lower-budget films migrated to streaming while only high-budget films maintained theatrical releases. Together, these findings demonstrate that financial and critical success in film are driven by fundamentally different factors, and that no single variable fully determines a film's outcome.

## Authors

- Kaly Nguyen: Conceptualization, Background research, Writing – original draft (Research Question, Hypothesis), Analysis
- Martin Monroy: Data curation, Software (GitHub setup, dataset merging), Writing – original draft (Abstract)
- Ketan Pandey: Data curation, Software (dataset merging, OLS model), Visualization, Writing – review & editing
- Jayline Sanchez: Analysis, Visualization, Writing – original draft (Discussion & Conclusion)
- Jacqueline Huynh: Analysis, Visualization, Writing – original draft (Discussion & Conclusion), Writing – review & editing

## Research Question

How do production budget, movie genre, lead actors, language, and country of production influence a film's financial and critical success — measured by box office revenue and IMDb ratings — and does this relationship differ between US and non-US productions, and across the pre- and post-streaming eras (before and after 2015)?

Specifically, we examine whether the budget-success relationship has weakened as streaming platforms have disrupted theatrical revenue models, and whether Hollywood's historically dominant global distribution network produces a stronger budget-revenue association compared to non-US film industries. We are also interested in whether lower-budget international films in certain genres can achieve comparable critical success (IMDb ratings) despite lower revenue, suggesting that financial and critical success are not always aligned.

The key variables include production budget (numerical), genre (categorical), country of production (categorical), lead actor (categorical), original language (categorical), release year (numerical, used to define pre/post-2015 periods), box office revenue (numerical), and IMDb rating (numerical). We will use correlation analysis, group comparisons, and interaction-based statistical models to examine these relationships.

## Background and Prior Work

The film industry is strongly influenced by production budgets, which affect everything from casting and visual effects to marketing and distribution. Different movie genres tend to require different levels of funding. Research on box office revenue prediction has confirmed that production budget is the single strongest predictor of a film's financial performance, with action and adventure genres consistently commanding the highest budgets while drama and horror operate at far lower cost levels.<sup>1</sup>

Previous peer-reviewed work has also shown that genre plays a major role in determining production cost and financial outcomes. Shahid and Islam (2023) demonstrated that a genre's popularity at the time of release is a strong predictor of box office returns, and that drama films consistently carry lower budgets than action or adventure films — underscoring how budget and genre interact to shape financial outcomes.<sup>2</sup>

When it comes to critical success, the picture is different. A study analyzing IMDb user voting data found no significant correlation between production budget and average user ratings, suggesting that audience satisfaction is driven by factors such as storytelling quality and emotional resonance rather than spending alone.<sup>3</sup> This supports our hypothesis that financial and critical success are not always aligned.

Despite this body of work, several important gaps remain. Most existing studies focus almost exclusively on Hollywood. However, industry data show that international revenues for US studio films now account for over 70% of Hollywood's total box office take, and that non-US industries operate under fundamentally different distribution conditions.<sup>4</sup> Additionally, research published in Applied Economics found that the launch of Netflix streaming in the US was associated with a 14–17% reduction in theatrical box office revenues.<sup>5</sup>

**References**

1. Joshi, A., et al. (2024). Movie Revenue Prediction Using Machine Learning Models. arXiv. https://arxiv.org/abs/2405.11651
2. Shahid, M. H., & Islam, M. A. (2023). PeerJ Computer Science, 9, e1603. https://doi.org/10.7717/peerj-cs.1603
3. Wasserman, M., et al. (2014). arXiv. https://arxiv.org/abs/1312.3986
4. Follows, S. (2017). https://stephenfollows.com/p/important-international-box-office-hollywood
5. Walls, W. D. (2024). Applied Economics, 57(44). https://doi.org/10.1080/00036846.2024.2387860

## Hypothesis

We hypothesize that movies with larger production budgets, globally recognized actors, and English-language releases will generally earn higher box office revenue due to wider international distribution and marketing reach. Specifically, we expect this budget-revenue relationship to be stronger for US films than for non-US films, because Hollywood benefits from an established global distribution and marketing infrastructure that amplifies returns on high-budget productions.

We further hypothesize that this budget-revenue association has weakened in the post-2015 period compared to the pre-2015 era, as the proliferation of streaming platforms has reduced audiences' dependence on theatrical releases.

In contrast, we expect IMDb ratings to remain largely independent of production budget regardless of country of production or time period. We also predict that some lower-budget international films, particularly in drama or thriller genres, may receive disproportionately high IMDb ratings relative to their revenue.

Among genres, we expect action, adventure, and fantasy films to show the strongest positive budget-revenue correlation, while drama and horror will show weaker or more variable correlations.

## Data

### Data Overview

We use two complementary datasets that together cover the variables needed to address our research question. Dataset 1 provides IMDb scores and gross revenue with a focus on the 1986–2016 period, while Dataset 2 extends coverage through 2017 and adds variables such as original language, cast, and popularity score.

**Combining the datasets:** We merge the two datasets on movie title and release year after standardizing column names.

### Dataset #1 — Movie Industry

- **Link:** https://www.kaggle.com/datasets/danielgrijalvas/movies/data
- **Observations:** 6,820 | **Variables:** 15
- **Key variables:** `budget`, `gross`, `score`, `genre`, `star`, `country`, `year`, `rating`, `runtime`
- **Shortcomings:** Missing/zero budget values for many films; coverage ends at 2016; single genre per film.

In [ ]:
# Run once after cloning to download raw data
import sys
sys.path.append('./modules')
import get_data

datafiles = [
    { 'url': 'https://raw.githubusercontent.com/danielgrijalvas/movies/master/movies.csv',
      'filename': 'movies.csv' },
    { 'url': 'https://raw.githubusercontent.com/utkarshx27/movies-dataset/main/movie_dataset.csv',
      'filename': 'movie_dataset.csv' }
]
get_data.get_raw(datafiles, destination_directory='data/00-raw/')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Load and clean Dataset 1
df1 = pd.read_csv('data/00-raw/movies.csv')
df2 = pd.read_csv('data/00-raw/movie_dataset.csv')

df1['release_date'] = pd.to_datetime(
    df1['released'].str.extract(r'^([^(]+)').squeeze().str.strip(), errors='coerce'
)
df1 = df1.drop(columns=['released', 'votes'])
str_cols = df1.select_dtypes(include='object').columns
df1[str_cols] = df1[str_cols].apply(lambda s: s.str.strip())
for col in ['budget', 'gross', 'runtime']:
    df1[col] = pd.to_numeric(df1[col], errors='coerce')
df1.loc[df1['budget'] == 0, 'budget'] = np.nan
df1.loc[df1['gross']  == 0, 'gross']  = np.nan
df1 = df1.dropna(subset=['budget'])

# CPI-U inflation adjustment to 2020 dollars
cpi = {
    1980:82.4,1981:90.9,1982:96.5,1983:99.6,1984:103.9,
    1985:107.6,1986:109.6,1987:113.6,1988:118.3,1989:124.0,
    1990:130.7,1991:136.2,1992:140.3,1993:144.5,1994:148.2,
    1995:152.4,1996:156.9,1997:160.5,1998:163.0,1999:166.6,
    2000:172.2,2001:177.1,2002:179.9,2003:184.0,2004:188.9,
    2005:195.3,2006:201.6,2007:207.3,2008:215.3,2009:214.5,
    2010:218.1,2011:224.9,2012:229.6,2013:233.0,2014:236.7,
    2015:237.0,2016:240.0,2017:245.1,2018:251.1,2019:255.7,
    2020:258.8
}
BASE_CPI = cpi[2020]
df1['cpi'] = df1['year'].map(cpi)
df1['budget_real'] = df1['budget'] * (BASE_CPI / df1['cpi'])
df1['gross_real']  = df1['gross']  * (BASE_CPI / df1['cpi'])
df1 = df1.drop(columns=['cpi'])
print('Dataset 1 shape after cleaning:', df1.shape)

### Dataset #2 — Movie Dataset: Budgets, Genres, Insights

- **Link:** https://www.kaggle.com/datasets/utkarshx27/movies-dataset
- **Observations:** 4,802 | **Variables:** 24
- **Key variables:** `budget`, `revenue`, `genres`, `original_language`, `production_countries`, `cast`, `popularity`, `vote_average`
- **Shortcomings:** Currency units unspecified; zero/missing budget and revenue entries.

In [ ]:
# Clean Dataset 2
new_movies2 = df2.drop(columns=['keywords', 'homepage', 'tagline'])
new_movies2 = new_movies2.dropna(subset=['genres', 'director', 'cast'])
print('Dataset 2 shape after cleaning:', new_movies2.shape)

In [ ]:
# Merge datasets on title + year
df1['merge_title'] = df1['name'].str.lower().str.strip()
df1['merge_year']  = df1['year']
df2['merge_title'] = df2['title'].str.lower().str.strip()
df2['merge_year']  = pd.to_datetime(df2['release_date'], errors='coerce').dt.year

df1_clean = df1.dropna(subset=['merge_title','merge_year']).copy()
df2_clean = df2.dropna(subset=['merge_title','merge_year']).copy()
df2_clean['merge_year'] = df2_clean['merge_year'].astype(int)

df1_cols = ['merge_title','merge_year','name','year','genre','rating',
            'score','director','star','country','budget','gross','company','runtime']
df2_cols = ['merge_title','merge_year','genres','original_language',
            'production_countries','popularity','revenue','vote_average','vote_count','cast']

df = pd.merge(
    df1_clean[df1_cols], df2_clean[df2_cols],
    on=['merge_title','merge_year'], how='inner', suffixes=('_ds1','_ds2')
)
df = df.drop(columns=['merge_title','merge_year'])
df['log_budget'] = np.log1p(df['budget'])
df['log_gross']  = np.log1p(df['gross'])
df['era']        = df['year'].apply(lambda y: 'Post-2015' if y >= 2015 else 'Pre-2015')
df['is_us']      = df['country'] == 'United States'
df['US_Status']  = df['is_us'].map({True: 'US', False: 'Non-US'})
df.to_csv('data/02-processed/merged_movies.csv', index=False)
print('Merged dataset shape:', df.shape)
df.head()

## Results

### Exploratory Data Analysis

The following EDA examines distributions of key variables and relationships between them across genre, country of production, and release era. For full EDA code see `02-EDACheckpoint.ipynb`.

In [ ]:
# Graph 1: Univariate distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
budget_data = df['budget'].dropna()
budget_bins = np.logspace(np.log10(budget_data.min()), np.log10(budget_data.max()), 50)
axes[0].hist(budget_data, bins=budget_bins, color='steelblue', edgecolor='white')
axes[0].set_xscale('log')
axes[0].set_xlabel('Production Budget (USD, log scale)')
axes[0].set_ylabel('Number of Movies')
axes[0].set_title(f'Distribution of Budget (n={df["budget"].notna().sum()})')
gross_data = df['gross'].dropna()
gross_bins = np.logspace(np.log10(gross_data.min()), np.log10(gross_data.max()), 50)
axes[1].hist(gross_data, bins=gross_bins, color='seagreen', edgecolor='white')
axes[1].set_xscale('log')
axes[1].set_xlabel('Gross Revenue (USD, log scale)')
axes[1].set_ylabel('Number of Movies')
axes[1].set_title(f'Distribution of Gross Revenue (n={df["gross"].notna().sum()})')
axes[2].hist(df['score'].dropna(), bins=30, color='coral', edgecolor='white')
axes[2].set_xlabel('IMDb Score (0-10)')
axes[2].set_ylabel('Number of Movies')
axes[2].set_title(f'Distribution of IMDb Score (n={df["score"].notna().sum()})')
plt.tight_layout()
plt.savefig('results/fig1_univariate.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Median budget: ${df['budget'].median():,.0f}")
print(f"Median gross:  ${df['gross'].median():,.0f}")
print(f"Median score:  {df['score'].median():.1f}")

**Graph 1 — Univariate Distributions**

Both budget and revenue follow approximately log-normal distributions with peaks around $30M for budget and $55M for gross. Revenue is more variable than budget, spanning roughly 6 log-scale bins versus 4 for budget. IMDb scores are approximately normally distributed and centered at 6.6.

**Low-budget outlier check:** Films like Paranormal Activity ($15K → $193M) and The Blair Witch Project ($60K → $248M) represent extreme ROI outliers that could influence regression trend lines.

In [ ]:
# Low-budget outlier check
df[df['budget'] < 100000][['name','year','budget','gross','genre']].sort_values('budget')

In [ ]:
# Graph 2: Genre Distribution
genre_counts = df['genre'].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='viridis', ax=ax)
ax.set_xlabel('Number of Movies')
ax.set_ylabel('Genre')
ax.set_title('Movie Genre Distribution in Merged Dataset')
for i, v in enumerate(genre_counts.values):
    ax.text(v + 10, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('results/fig2_genre_dist.png', dpi=100, bbox_inches='tight')
plt.show()

**Graph 2 — Genre Distribution**

Action (905) and Comedy (849) together make up over 50% of the data. Genres like Fantasy (22), Mystery (8), and Sci-Fi (2) have too few observations for reliable group-level inference and are excluded from genre comparisons.

In [ ]:
# Graph 3: Budget vs Gross Revenue by Genre (log-log)
top_genres = df['genre'].value_counts().head(6).index.tolist()
df_plot = df[df['genre'].isin(top_genres)].dropna(subset=['budget','gross'])
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df_plot, x='budget', y='gross', hue='genre',
                alpha=0.6, s=30, palette='tab10', ax=ax)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Production Budget (USD, log scale)')
ax.set_ylabel('Gross Revenue (USD, log scale)')
ax.set_title('Budget vs. Gross Revenue by Genre (Top 6 Genres)')
xlim = ax.get_xlim()
ax.plot(xlim, xlim, 'k--', alpha=0.4, label='Break-even (Gross = Budget)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('results/fig3_budget_gross.png', dpi=100, bbox_inches='tight')
plt.show()
log_corr = np.log10(df_plot['budget']).corr(np.log10(df_plot['gross']))
print(f'Overall log-log correlation (budget vs gross): r = {log_corr:.3f}')

**Graph 3 — Budget vs. Gross Revenue (log-log, by genre)**

Strong positive log-log correlation (r = 0.673). Action films dominate the upper right quadrant. Films below the break-even line are concentrated at lower budget levels, suggesting smaller films are more prone to underperformance.

In [ ]:
# Graph 4: Correlation Heatmap
corr_matrix = df[['budget','gross','score']].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, square=True)
plt.title('Correlation Matrix: Budget, Gross Revenue, and IMDb Score')
plt.tight_layout()
plt.savefig('results/fig4_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

**Graph 4 — Correlation Heatmap**

Budget and gross are strongly correlated (r ≈ 0.74). Budget shows essentially zero correlation with IMDb score (r ≈ 0.07), strongly supporting the hypothesis that financial and critical success are driven by different factors.

In [ ]:
# Graph 5: Gross Revenue by Genre
top8 = df['genre'].value_counts().head(8).index.tolist()
df_top8 = df[df['genre'].isin(top8)].dropna(subset=['gross'])
order = df_top8.groupby('genre')['gross'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_top8, x='genre', y='gross', order=order,
            palette='Set2', fliersize=2, ax=ax)
ax.set_yscale('log')
ax.set_xlabel('Genre'); ax.set_ylabel('Gross Revenue (USD, log scale)')
ax.set_title('Gross Revenue Distribution by Genre (Top 8 Genres, sorted by median)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('results/fig5_gross_genre.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Graph 6: IMDb Score by Genre
df_score = df[df['genre'].isin(top8)].dropna(subset=['score'])
score_order = df_score.groupby('genre')['score'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_score, x='genre', y='score', order=score_order,
            palette='Set3', fliersize=2, ax=ax)
ax.set_xlabel('Genre'); ax.set_ylabel('IMDb Score (0-10)')
ax.set_title('IMDb Score Distribution by Genre (Top 8 Genres, sorted by median)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('results/fig6_score_genre.png', dpi=100, bbox_inches='tight')
plt.show()
print('Median IMDb score by genre:')
print(df_score.groupby('genre')['score'].median().sort_values(ascending=False).round(2))

**Graphs 5 & 6 — Revenue and IMDb Score by Genre**

Animation, Adventure, and Action show the highest median revenues. Strikingly, Drama and Biography — low revenue earners — receive the highest IMDb ratings. This is direct evidence that financial and critical success are not aligned.

In [ ]:
# Graph 7: US vs Non-US
g = sns.lmplot(data=df, x='log_budget', y='log_gross', hue='is_us',
               scatter_kws={'alpha': 0.35, 's': 15}, height=6, aspect=1.2)
plt.xlabel('Log Budget'); plt.ylabel('Log Gross Revenue')
plt.title('Budget-Revenue Relationship: US vs. Non-US Films')
plt.savefig('results/fig7_us_nonus.png', dpi=100, bbox_inches='tight')
plt.show()
us_corr    = df[df['is_us']]['log_budget'].corr(df[df['is_us']]['log_gross'])
nonus_corr = df[~df['is_us']]['log_budget'].corr(df[~df['is_us']]['log_gross'])
print(f'US films log-log correlation:     r = {us_corr:.3f}')
print(f'Non-US films log-log correlation: r = {nonus_corr:.3f}')

In [ ]:
# Graph 8: Pre/Post-2015
g = sns.lmplot(data=df, x='log_budget', y='log_gross', hue='era',
               scatter_kws={'alpha': 0.35, 's': 15}, height=6, aspect=1.2)
plt.xlabel('Log Budget'); plt.ylabel('Log Gross Revenue')
plt.title('Budget-Revenue Relationship by Era (Pre vs. Post-2015)')
plt.savefig('results/fig8_era.png', dpi=100, bbox_inches='tight')
plt.show()
pre_corr  = df[df['era']=='Pre-2015']['log_budget'].corr(df[df['era']=='Pre-2015']['log_gross'])
post_corr = df[df['era']=='Post-2015']['log_budget'].corr(df[df['era']=='Post-2015']['log_gross'])
print(f'Pre-2015 correlation:  r = {pre_corr:.3f}')
print(f'Post-2015 correlation: r = {post_corr:.3f}')

In [ ]:
# Graph 9: Budget vs IMDb Score
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df, x='log_budget', y='score', alpha=0.4, s=15, ax=ax)
sns.regplot(data=df, x='log_budget', y='score', scatter=False, color='red',
            line_kws={'linewidth': 2}, ax=ax)
ax.set_xlabel('Log Budget'); ax.set_ylabel('IMDb Score (0-10)')
ax.set_title('Production Budget vs. IMDb Rating')
plt.tight_layout()
plt.savefig('results/fig9_budget_score.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Budget-IMDb correlation: r = {df["log_budget"].corr(df["score"]):.3f}')

**Graphs 7, 8, 9 — US vs Non-US, Era, and Budget vs IMDb Score**

US films have a stronger budget-revenue correlation (r ≈ 0.67) than non-US films (r ≈ 0.59). Contrary to hypothesis, post-2015 correlation is slightly higher than pre-2015, likely due to a selection effect where lower-budget films moved to streaming. Budget shows essentially zero correlation with IMDb score (r ≈ 0.00), confirming that spending more does not produce better-reviewed films.

The goal of this EDA was to better understand the relationship between movie success and production characteristics. Results show that action movies often brought in the most gross revenue. US films were much more successful in gross revenue. What is interesting is that there is little to no relationship between IMDb ratings and movie budgets — the budget used to create a movie does not have much impact on audience ratings. Additionally, genres like drama and biography tended to have higher IMDb ratings even though they were not always the highest-earning films. Post-2015 films demonstrated stronger revenue performance, potentially reflecting the increasing globalization of film distribution. However, audience ratings did not increase at the same rate, reinforcing the idea that higher earnings do not automatically indicate greater audience satisfaction.

### Descriptive Analysis

Before running inferential analyses, we first get a high-level sense of the merged dataset's structure and key statistics.

In [ ]:
# Shape of the merged dataset
print('Dataset shape:', df.shape)

# Descriptive statistics for key numeric variables
df[['budget', 'gross', 'score', 'runtime']].describe().round(2)

In [ ]:
# Data types and missing value summary
print('Data types:')
print(df[['budget','gross','score','genre','country','year','is_us','era']].dtypes)
print('\nMissing values:')
print(df[['budget','gross','score','genre','country','year']].isnull().sum())

The merged dataset contains 3,000+ films with key numeric variables including `budget`, `gross`, and `score`. Budget and gross are heavily right-skewed — median budget is around $30M while the mean is much higher, confirming the log-normal distribution observed in EDA. IMDb scores are approximately normally distributed with a median of 6.6. There are missing gross values for some films (unreported theatrical revenue), which we handle by dropping rows with missing values before regression.

### Analysis 1: OLS Multiple Regression — Predicting Box Office Gross

To formally test how budget, genre, country of origin, release era, and IMDb score jointly predict box office performance, we fit an OLS multiple regression model with log gross revenue as the outcome. Log-transforming financial variables is appropriate because both budget and gross follow log-normal distributions confirmed in EDA.

**Null hypothesis (H₀):** Log budget, US status, era, genre, and IMDb score have no effect on log gross revenue.  
**Alternative hypothesis (H₁):** At least one predictor has a significant effect on log gross revenue.

In [ ]:
# Analysis 1: OLS Multiple Regression
df_reg = df.dropna(subset=['log_budget','log_gross','score','genre','is_us','era'])

model = smf.ols(
    """
    log_gross ~ log_budget
              + C(is_us)
              + C(era)
              + log_budget:C(is_us)
              + log_budget:C(era)
              + C(genre)
              + score
    """,
    data=df_reg
).fit()

print(model.summary())

In [ ]:
# Residuals vs Fitted plot
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(model.fittedvalues, model.resid, alpha=0.4, s=15, color='steelblue')
ax.axhline(0, color='red', linewidth=1.2, linestyle='--')
ax.set_xlabel('Fitted log(Gross)')
ax.set_ylabel('Residuals')
ax.set_title('OLS Residuals vs. Fitted Values')
plt.tight_layout()
plt.savefig('results/fig10_ols_residuals.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'R-squared:       {model.rsquared:.3f}')
print(f'Adj. R-squared:  {model.rsquared_adj:.3f}')
print(f'N observations:  {int(model.nobs)}')

**Regression Results & Interpretation**

The OLS model explains approximately 50% of the variance in log gross revenue (R² ≈ 0.50).

- **Log budget** is the dominant predictor (coef ≈ 0.87, p < 0.001). A 1% increase in production budget is associated with roughly a 0.87% increase in gross revenue.
- **US films** earn significantly more than non-US films (p < 0.05). The interaction term is not significant — the slope does not differ by country once other factors are controlled.
- **Era:** Neither the era main effect nor its interaction is significant after controlling for genre and budget.
- **IMDb score** has a small but significant positive effect (coef ≈ 0.20, p < 0.001): better-reviewed films earn slightly more, but far less than budget explains.
- The residual plot shows no major systematic pattern, supporting the appropriateness of a linear model in log space.

### Analysis 2: Welch's t-test — Do US Films Earn Significantly More Than Non-US Films?

We directly test the US vs. non-US revenue gap using a Welch's t-test on log-transformed gross revenue. Welch's t-test does not assume equal variances between groups, making it appropriate here.

**Null hypothesis (H₀):** The mean log gross revenue of US films equals that of non-US films.  
**Alternative hypothesis (H₁):** US films have a higher mean log gross revenue than non-US films (one-tailed).

In [ ]:
# Analysis 2: Welch's t-test — US vs Non-US log gross
df_ttest = df.dropna(subset=['log_gross','is_us'])

us_log_gross    = df_ttest[df_ttest['is_us'] == True]['log_gross']
nonus_log_gross = df_ttest[df_ttest['is_us'] == False]['log_gross']

t_stat, p_val_two = stats.ttest_ind(us_log_gross, nonus_log_gross, equal_var=False)
p_val_one = p_val_two / 2  # one-tailed

print(f'US films     — n={len(us_log_gross)}, mean log gross={us_log_gross.mean():.3f}, std={us_log_gross.std():.3f}')
print(f'Non-US films — n={len(nonus_log_gross)}, mean log gross={nonus_log_gross.mean():.3f}, std={nonus_log_gross.std():.3f}')
print(f'\nWelch t-statistic:  {t_stat:.3f}')
print(f'Two-tailed p-value: {p_val_two:.4f}')
print(f'One-tailed p-value: {p_val_one:.4f}')

In [ ]:
# Visualization: log gross distribution by US status
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(us_log_gross, bins=40, alpha=0.6, color='steelblue',
        label=f'US (n={len(us_log_gross)})', density=True)
ax.hist(nonus_log_gross, bins=40, alpha=0.6, color='coral',
        label=f'Non-US (n={len(nonus_log_gross)})', density=True)
ax.axvline(us_log_gross.mean(), color='steelblue', linestyle='--', linewidth=1.8,
           label=f'US mean = {us_log_gross.mean():.2f}')
ax.axvline(nonus_log_gross.mean(), color='coral', linestyle='--', linewidth=1.8,
           label=f'Non-US mean = {nonus_log_gross.mean():.2f}')
ax.set_xlabel('Log Gross Revenue')
ax.set_ylabel('Density')
ax.set_title('Distribution of Log Gross Revenue: US vs. Non-US Films')
ax.legend()
plt.tight_layout()
plt.savefig('results/fig11_ttest.png', dpi=100, bbox_inches='tight')
plt.show()

**t-test Results & Interpretation**

The Welch's t-test yields a statistically significant result (p < 0.001, one-tailed), allowing us to reject the null hypothesis. US films earn significantly higher log gross revenues than non-US films on average.

This result is consistent with our hypothesis that Hollywood's established global distribution infrastructure produces a revenue advantage beyond what budget alone can explain, and aligns with the OLS regression results where the US indicator was a significant positive predictor even after controlling for budget, genre, and IMDb score.

**Caveat:** Non-US films are substantially underrepresented in our merged dataset, which may inflate the measured gap. Results for non-US subgroups should be interpreted cautiously.

## Ethics

### A. Data Collection
- **A.1 Informed consent:** This project uses publicly available datasets (IMDb via Kaggle), so no human subjects are directly involved and informed consent is not required.
- **A.2 Collection bias:** Both datasets are biased toward popular, Western, and high-budget films, which may underrepresent non-US industries such as Bollywood, Nollywood, or Korean cinema. We explicitly report sample sizes by country group and acknowledge this imbalance when interpreting results.
- **A.3 Limit PII exposure:** The datasets do not include personally identifiable information. All data points represent films, not individuals.
- **A.4 Downstream bias mitigation:** We flag limitations and avoid overgeneralizing from underrepresented groups.

### B. Data Storage
- **B.1 Data security:** Data is stored in a private GitHub repository accessible only to team members.
- **B.2 Right to be forgotten:** All data used is publicly available and contains no personal information.
- **B.3 Data retention plan:** Data will only be used for this academic project and deleted upon course completion.

### C. Analysis
- **C.1 Missing perspectives:** Our datasets significantly underrepresent non-English-language and non-Western film industries.
- **C.2 Dataset bias:** We examine potential bias in genre representation and budget reporting.
- **C.3 Honest representation:** All visualizations and conclusions accurately reflect the underlying data. We report null and unexpected findings alongside positive ones.
- **C.4 Privacy in analysis:** No personal data is displayed or used.
- **C.5 Auditability:** All analysis is documented in reproducible Jupyter notebooks, version-controlled on GitHub.

### D. Modeling
- **D.1 Proxy discrimination:** Country of production and original language are used as structural variables, not as proxies for race or ethnicity.
- **D.2 Fairness across groups:** We report results separately for US vs. non-US films and for pre/post-2015 periods.
- **D.3 Metric selection:** We acknowledge that box office revenue does not capture streaming income or home video sales.
- **D.4 Explainability:** Methods and results are clearly explained in plain language alongside technical details.
- **D.5 Communicate limitations:** We explicitly state dataset limitations, potential sources of bias, and boundaries on generalizability.

### E. Deployment
- **E.1 Monitoring and evaluation:** This is an academic project with no ongoing deployment.
- **E.2 Redress:** If errors are identified, we will revise our analysis and clearly document what changed.
- **E.3 Roll back:** All analysis is version-controlled on GitHub.
- **E.4 Unintended use:** We communicate findings responsibly with appropriate caveats.

## Discussion and Conclusion

Overall, after conducting our analysis we were able to determine the existing factors that impact a movie's success. Our results showed that higher production budgets had a large influence on the financial success of films, confirmed both by the strong log-log correlation (r ≈ 0.67) in EDA and by the OLS regression coefficient (≈ 0.87, p < 0.001) which held even after controlling for genre, country, era, and IMDb score. However, high production budgets were found to have no impact on IMDb scores, suggesting that audiences may value originality and storytelling more than how costly the film was. Genre also plays a part in gross revenue, with animation, action, and adventure earning the most. Budget-revenue associations were not uniform across genres — drama and biography earned significantly less. Our Welch's t-test confirmed that US films earn significantly higher gross revenues than non-US films (p < 0.001), consistent with Hollywood's established global distribution advantage. Lastly, post-2015 films showed a slightly stronger budget-revenue relationship rather than a weaker one as hypothesized, which we attribute to a selection effect: lower-budget films increasingly migrated to streaming platforms post-2015, leaving theatrical data dominated by high-budget wide releases.

The film industry is an ever-changing phenomenon that drives popular culture. The definition of a "successful" film cannot be explained by a single indicator. Financial success was most strongly associated with production-related characteristics such as budget, genre, release period, and country of origin. However, these factors did not consistently predict audience reaction and appreciation. IMDb ratings displayed a weak relationship with budget and genres. Drama and Biography often received higher ratings despite earning less revenue than blockbuster genres, emphasizing the contradiction between commercial and critical success. While high-grossing films may owe part of their success to substantial marketing budgets and widespread appeal, highly rated films often resonate with audiences through compelling stories, creativity, or memorable visuals. Success in film is not measured solely by revenue — many movies that underperform financially still leave an enduring mark on audiences and popular culture.

Our analysis has several limitations. First, both datasets are heavily skewed toward Hollywood and English-language films, making it difficult to draw strong conclusions about the global film industry as a whole. In addition, we were forced to remove observations with missing or unreported budget information, which may have systematically excluded smaller or non-US productions. Moreover, some genres had very small sample sizes. Finally, our data focuses primarily on theatrical revenue and does not account for streaming revenue, licensing deals, or other modern sources of income. If we were to continue this project, we would like to include more international films and incorporate streaming performance data. It would also be interesting to examine additional factors such as marketing budgets, franchise status, awards recognition, or audience demographics.